# Phase 1: Antarctic Sea-Ice Data Exploration

## SIH PS 59

### Dataset: NOAA/NSIDC Sea Ice Index, Version 3 (G02135)
- DOI: 10.7265/N5K072F8
- Source: https://noaadata.apps.nsidc.org/NOAA/G02135/
- Variables: extent, area (million km2)
- Temporal: 1978-2026, monthly
- Region: Antarctic

## Section 2: Imports

In [5]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from src.data.load import load_sea_ice, get_dataset_metadata
from src.data.validate import validate_dataset
from src.data.preprocess import preprocess_sea_ice
from src.data.geo import antarctic_bbox, haversine_distance

print("Imports OK")

Imports OK


## Section 2b: Antarctic Base Map

In [ ]:
from src.data.geo import create_antarctic_base_map
import matplotlib
matplotlib.use("Agg")

fig, ax = create_antarctic_base_map(title="Antarctic Region - Phase 1 Base Map")
plt.tight_layout()
plt.savefig("../data/processed/antarctic_base_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Base map saved to data/processed/antarctic_base_map.png")

## Section 3: Dataset Loading

In [6]:
from src.data.download import download_g02135_monthly
download_g02135_monthly()

df_raw = load_sea_ice()
print(f"Loaded: {len(df_raw)} records")
df_raw.head(10)

  [EXISTS] S_01_extent_v4.0.csv
  [EXISTS] S_02_extent_v4.0.csv
  [EXISTS] S_03_extent_v4.0.csv
  [EXISTS] S_04_extent_v4.0.csv
  [EXISTS] S_05_extent_v4.0.csv
  [EXISTS] S_06_extent_v4.0.csv
  [EXISTS] S_07_extent_v4.0.csv
  [EXISTS] S_08_extent_v4.0.csv
  [EXISTS] S_09_extent_v4.0.csv
  [EXISTS] S_10_extent_v4.0.csv
  [EXISTS] S_11_extent_v4.0.csv
  [EXISTS] S_12_extent_v4.0.csv
Loaded: 573 records


,year,mo,source_dataset,region,extent,area
0,1978,11,NSIDC-0051,S,15.90,11.69
1,1978,12,NSIDC-0051,S,10.40,6.97
2,1979,1,NSIDC-0051,S,5.40,3.47
3,1979,2,NSIDC-0051,S,3.14,2.11
4,1979,3,NSIDC-0051,S,4.00,2.66
5,1979,4,NSIDC-0051,S,7.49,5.45
6,1979,5,NSIDC-0051,S,10.83,8.30
7,1979,6,NSIDC-0051,S,14.19,11.22
8,1979,7,NSIDC-0051,S,16.52,13.26
9,1979,8,NSIDC-0051,S,17.70,13.79


## Section 4: Metadata Exploration

In [7]:
metadata = get_dataset_metadata(df_raw)
for key, value in metadata.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

dimensions:
  n_records: 573
  n_years: 49
  n_months: 12
variables: ['year', 'mo', 'source_dataset', 'region', 'extent', 'area']
time_range:
  start_year: 1978
  end_year: 2026
  start_month: 11
  end_month: 1
spatial_coverage:
  region: ['S']
  description: Antarctic (S) and/or Arctic (N) aggregate statistics
units:
  extent: million km^2
  area: million km^2
source_dataset: ['NSIDC-0051', '-9999', 'NSIDC-0051,NSIDC-0081', 'NSIDC-0803']
missing_values:
  extent: 0
  area: 0
value_ranges:
  extent_min: -9999.0
  extent_max: 19.76
  area_min: -9999.0
  area_max: 15.75
doi: 10.7265/N5K072F8
citation: Fetterer, F., Knowles, K., Meier, W. N., Savoie, M. & Windnagel, A. K. (2017). Sea Ice Index. (G02135, Version 3). [Data Set]. Boulder, Colorado USA. National Snow and Ice Data Center.


## Section 5: Validation

In [8]:
result = validate_dataset(df_raw, dataset_type="g02135")
print("Overall:", result["overall"])
print()
for check in result["checks"]:
    s = check["status"]
    sym = "[PASS]" if s=="PASS" else "[WARN]" if s=="WARN" else "[FAIL]"
    print(f"  {sym} {check['check']}: {check['detail']}")

Overall: FAIL
  [PASS] Dataset loaded: Type=DataFrame, rows=573
  [PASS] Required columns: Found: {'year', 'mo', 'region', 'area', 'extent', 'source_dataset'}
  [PASS] Year validity (1978-2030): Range: 1978-2026
  [PASS] Month validity (1-12): Unique months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
  [WARN] Extent values valid (> 0, < 30 M km2): Range: -9999.00-19.76 M km2
  [WARN] Area values valid (> 0, < 30 M km2): Range: -9999.00-15.75 M km2
  [PASS] Area <= Extent (physical constraint): Violations: 0
  [PASS] Missing values: Missing: 0 values in extent/area
  [PASS] Region is Antarctic (S): Regions: ['S']


## Section 6: Preprocessing

In [9]:
processed = preprocess_sea_ice(df_raw, region="S")
print(f"Processed: {len(processed)} records")
print(f"Date range: {processed.date.min()} to {processed.date.max()}")
processed.head(10)

  [INFO] Dropped 2 rows with fill values (missing data)
Processed: 571 records
Date range: 1978-11-01 00:00:00 to 2026-07-01 00:00:00


,year,month,extent,area,date,ice_efficiency
0,1978,11,15.90,11.69,1978-11-01,0.735220
1,1978,12,10.40,6.97,1978-12-01,0.670192
2,1979,1,5.40,3.47,1979-01-01,0.642593
3,1979,2,3.14,2.11,1979-02-01,0.671975
4,1979,3,4.00,2.66,1979-03-01,0.665000
5,1979,4,7.49,5.45,1979-04-01,0.727637
6,1979,5,10.83,8.30,1979-05-01,0.766390
7,1979,6,14.19,11.22,1979-06-01,0.790698
8,1979,7,16.52,13.26,1979-07-01,0.802663
9,1979,8,17.70,13.79,1979-08-01,0.779096


## Section 7: EDA

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(processed["extent"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Antarctic Sea-Ice Extent Distribution")
axes[0].set_xlabel("Extent (M km2)")
axes[1].hist(processed["area"], bins=50, color="coral", edgecolor="white")
axes[1].set_title("Antarctic Sea-Ice Area Distribution")
axes[1].set_xlabel("Area (M km2)")
plt.tight_layout()
plt.savefig("../data/processed/extent_area_distribution.png", dpi=150)
print("Saved extent_area_distribution.png")

Saved sic_distribution.png


In [11]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(processed["date"], processed["extent"], label="Extent", color="steelblue")
ax.plot(processed["date"], processed["area"], label="Area", color="coral")
ax.set_title("Antarctic Sea-Ice Extent and Area (1978-2026)")
ax.set_ylabel("Million km2")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../data/processed/extent_area_timeseries.png", dpi=150)
print("Saved extent_area_timeseries.png")

Saved antarctic_sic_timeseries.png


In [12]:
clim = processed.groupby("month")[["extent", "area"]].mean()
fig, ax = plt.subplots(figsize=(10, 6))
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ax.plot(range(1,13), clim["extent"], "o-", color="steelblue", label="Extent")
ax.plot(range(1,13), clim["area"], "s-", color="coral", label="Area")
ax.set_xticks(range(1,13))
ax.set_xticklabels(months)
ax.set_title("Monthly Climatology (Extent & Area)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../data/processed/extent_area_climatology.png", dpi=150)
print("Saved climatology.png")

Saved climatology.png


## Section 8: Save Processed Dataset

In [13]:
processed.to_csv("../data/processed/extent_area_monthly.csv", index=False)
print("Saved extent_area_monthly.csv")
df_check = pd.read_csv("../data/processed/extent_area_monthly.csv")
print(f"Reopened: {len(df_check)} records, columns: {list(df_check.columns)}")

Saved antarctic_sic_monthly.csv
Reopened: 571 records, columns: ['year', 'month', 'extent', 'area', 'date', 'ice_efficiency']


## Phase 1 Complete - Ready for Phase 2

## Phase 2 Data Requirement

The G02135 dataset provides Antarctic sea-ice **extent** and **area** time series (aggregate scalars).

It does **not** provide the spatial sea-ice concentration field required for Phase 2 forecasting.

**Phase 2 will integrate** an appropriate real spatial Antarctic Sea-Ice Concentration dataset
for ML forecasting (e.g., NOAA/NSIDC CDR G02202 or NSIDC Sea Ice Index gridded products).

Do not treat G02135 extent/area as spatial SIC.